# MariTerm 2 IWN

Importing libraries and defining main variables

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import csv
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import string
import xml.etree.ElementTree as ET
from xml.dom import minidom
import xml.dom.minidom

In [ ]:
MariT ='/content/drive/MyDrive/Desktop/MariT2IWN/MariT_03_24.xml'
ItalWN = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_03_24.xml'
candidates_csv = '/content/drive/MyDrive/Desktop/MariT2IWN/candidates_06_24.csv'
breakdown_csv = '/content/drive/MyDrive/Desktop/MariT2IWN/breakdown_07_24.csv'
filt_mart = '/content/drive/MyDrive/Desktop/MariT2IWN/MariT_07_f_24.xml'
filt_iwn = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_07_f_24.xml'
updates = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_08a_upd_24.xml'
IWN_pre_mod = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_08b_pre_m_24.xml'
IWN_post_mm = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_09a_post_m_24.xml'
IWN_mm_w_glosses = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_09b_pm_and_glosses.xml'
finalized_IWN = '/content/drive/MyDrive/Desktop/MariT2IWN/IWN_10_24.xml'
finalized_MariT = '/content/drive/MyDrive/Desktop/MariT2IWN/MariT_10_24.xml'

allowed_relation_types = {"has_hyperonym", "has_hyponym", "near_synonym", "has_xpos_hyperonym", "has_xpos_hyponym", "xpos_near_synonym"}

def parse_xml(file_path):
    return ET.parse(file_path).getroot()

# Part 1
A. Calculation of similarities between pairs of items

B. Saving metadata in csv format

C. Saving the matched entries in filtered XML


In [ ]:
def normalize_lemma(lemma):
    """Replace underscores with spaces in lemma, replace apostrophes with spaces, and lowercase the text."""
    return lemma.replace('_', ' ').lower().replace("'", " ").translate(str.maketrans('', '', string.punctuation.replace("'", ""))) if lemma else ""

def normalize_text(text):
    """Lowercase, replace underscores with spaces, replace apostrophes with spaces, and remove other punctuation from text."""
    text = text.lower().replace('_', ' ').replace("'", " ")  # Replace underscores and apostrophes with spaces
    return text.translate(str.maketrans('', '', string.punctuation.replace("'", "")))  # Remove all other punctuation

def extract_word_meanings(xml_root):
    word_meanings = []
    for word_meaning in xml_root.findall(".//WORD_MEANING"):
        lemma_variants = word_meaning.find("VARIANTS")
        lemmas = [literal.get("LEMMA") for literal in lemma_variants.findall("LITERAL") if literal.get("LEMMA")]
        normalized_lemmas = [normalize_lemma(lemma) for lemma in lemmas]

        senses = [literal.get("SENSE") for literal in lemma_variants.findall("LITERAL") if literal.get("SENSE")]
        normalized_senses = [normalize_text(sense) for sense in senses]

        gloss = word_meaning.find("GLOSS").text or ""
        normalized_gloss = normalize_text(gloss)

        relations = []
        for relation in word_meaning.findall(".//INTERNAL_LINKS/RELATION"):
            target_wm = relation.find("TARGET_WM")
            if target_wm is not None:
                target_id = target_wm.get("ID")
                target_lemma = target_wm.get("LEMMA")
                normalized_target_lemma = normalize_lemma(target_lemma)

                target_gloss = target_wm.get("GLOSS", "No Gloss")
                normalized_target_gloss = normalize_text(target_gloss)

                relation_type = relation.get("TYPE")
                if relation_type in allowed_relation_types:
                    relations.append({
                        "relation_type": relation_type,
                        "target_id": target_id,
                        "target_lemma": target_lemma,
                        "normalized_target_lemma": normalized_target_lemma,
                        "target_gloss": target_gloss,
                        "normalized_target_gloss": normalized_target_gloss
                    })

        first_literal = lemma_variants.find("LITERAL")
        first_literal_lemma = first_literal.get("LEMMA") if first_literal is not None else "No Literal Lemma"
        normalized_first_literal_lemma = normalize_lemma(first_literal_lemma)

        first_literal_sense = first_literal.get("SENSE") if first_literal is not None else "No Sense"
        normalized_first_literal_sense = normalize_text(first_literal_sense)

        word_meanings.append({
            "id": word_meaning.get("ID"),
            "lemmas": lemmas,
            "normalized_lemmas": normalized_lemmas,
            "senses": senses,
            "normalized_senses": normalized_senses,
            "gloss": gloss,
            "normalized_gloss": normalized_gloss,
            "relations": relations,
            "first_literal_lemma": first_literal_lemma,
            "normalized_first_literal_lemma": normalized_first_literal_lemma,
            "first_literal_sense": first_literal_sense,
            "normalized_first_literal_sense": normalized_first_literal_sense,
            "part_of_speech": word_meaning.get("PART_OF_SPEECH")
        })
    return word_meanings

def calculate_gloss_similarity(text1, text2):
    text1 = normalize_text(text1)
    text2 = normalize_text(text2)
    if not text1.strip() or not text2.strip():
        return 0.0

    vectorizer = TfidfVectorizer()
    try:
        tfidf = vectorizer.fit_transform([text1, text2])
        return cosine_similarity(tfidf[0:1], tfidf[1:2])[0][0]
    except ValueError as e:
        print(f"Error calculating gloss similarity: {e}")
        return 0.0

def calculate_relation_similarity(wn_g_sim, relations1, relations2):
    weights = {
        "has_hyperonym": 0.82,
        "has_hyponym": 0.76,
        "near_synonym": 0.6,
        "has_xpos_hyperonym": 0.5,
        "has_xpos_hyponym": 0.5,
        "xpos_near_synonym": 0.5
    }

    relation_weights = {}
    total_weighted_similarity = 0
    bonus = 0
    malus = 0
    bonus_relations = []
    malus_count = 0
    missing_relations_details = []
    no_gloss_relations = []
    no_gloss_words = []

    relations1_dict = {(r["relation_type"], r["target_lemma"]): r["target_gloss"] for r in relations1}
    relations2_dict = {(r["relation_type"], r["target_lemma"]): r["target_gloss"] for r in relations2}

    for key, mari_gloss in relations1_dict.items():
        if key in relations2_dict:
            ital_gloss = relations2_dict[key]

            if (mari_gloss != "No Gloss" and ital_gloss == "No Gloss") or (mari_gloss == "No Gloss" and ital_gloss != "No Gloss"):
                gloss_similarity = 0.5
            else:
                gloss_similarity = calculate_gloss_similarity(mari_gloss, ital_gloss)

            weight = weights.get(key[0], 0)
            weighted_similarity = gloss_similarity * weight

            relation_weights[key] = (weighted_similarity, gloss_similarity, weight)
            total_weighted_similarity += weighted_similarity

            if (mari_gloss != "No Gloss" and ital_gloss != "No Gloss") or (mari_gloss == "No Gloss" and ital_gloss == "No Gloss"):
                bonus += 0.33
                bonus_relations.append(key[1])
            elif mari_gloss != "No Gloss" and ital_gloss == "No Gloss":
                malus -= 0.33
                no_gloss_relations.append(key[1])
                no_gloss_words.append(mari_gloss)
        else:
            malus -= 0.33
            missing_relations_details.append(key[1])

    total_similarity = wn_g_sim + total_weighted_similarity + bonus + malus
    malus_count = len(no_gloss_relations)

    return total_weighted_similarity, relation_weights, bonus, malus, malus_count, missing_relations_details, no_gloss_relations, no_gloss_words, total_similarity, bonus_relations

def format_relations(relations, relation_weights, show_scores=True):
    formatted_relations = []
    seen_relations = set()

    for rel in relations:
        if isinstance(rel, dict):
            rel_type = rel["relation_type"]
            target_lemma = rel["target_lemma"]
            target_gloss = rel.get("target_gloss", "No Gloss")
            key = (rel_type, target_lemma)

            if key not in seen_relations:
                seen_relations.add(key)
                if key in relation_weights and show_scores:
                    weight_score, gloss_similarity, weight = relation_weights[key]
                    formatted_relations.append(f"{rel_type}, {target_lemma}, {target_gloss} "
                        f"(Score: [{gloss_similarity:.2f}] * [{weight:.2f}] = {weight_score:.2f})")
                else:
                    formatted_relations.append(f"{rel_type}, {target_lemma}, {target_gloss} (Score: 0.00)")

    return formatted_relations

def get_fallback_gloss(term, fallback_type='near_synonym'):
    # If a Word Meaning does not have a gloss, a fallback one is used. Check the primary fallback type first
    for relation_type in [fallback_type, 'near_xpos_synonym', 'has_hyponym', 'has_hyperonym']:
        for rel in term.get('relations', []):
            if rel.get('relation_type') == relation_type:
                target_gloss = rel.get('target_gloss', '')
                if target_gloss == "No Gloss":
                    target_gloss = ''  # Replace "No Gloss" with an empty string for comparison
                if target_gloss:
                    return target_gloss, relation_type
    return '', None  # If no valid fallback is found

def match_lemmas_with_alternate_senses(mari_term_wms, ital_wn_wms):
    results = []
    best_matches_per_ital_wn = {}
    seen_pairs = set()

    if not mari_term_wms or not ital_wn_wms:
        return results

    for wm1 in mari_term_wms:
        best_match = None
        best_gloss_similarity = -1
        best_total_weighted_relation_similarity = -1
        best_total_similarity = -1

        for wm2 in ital_wn_wms:
            if wm1.get("normalized_first_literal_lemma") == wm2.get("normalized_first_literal_lemma") and wm1.get("part_of_speech") == wm2.get("part_of_speech"):
                mari_gloss = wm1.get("normalized_gloss", "")
                ital_gloss = wm2.get("normalized_gloss", "")

                # Handle Mariterm fallback gloss
                if not mari_gloss.strip():
                    mari_gloss, mari_fallback_type = get_fallback_gloss(wm1, 'near_synonym')
                    fallback_used = True
                else:
                    mari_fallback_type = None
                    fallback_used = False

                # Handle ItalWN fallback gloss
                if not ital_gloss.strip():
                    ital_gloss, ital_fallback_type = get_fallback_gloss(wm2, 'near_synonym')
                    ital_fallback_used = True
                    if ital_gloss == 'No Gloss':
                        ital_gloss = ''  # Compare against an empty string instead of "No Gloss"
                else:
                    ital_fallback_type = None
                    ital_fallback_used = False

                wn_g_sim = calculate_gloss_similarity(mari_gloss, ital_gloss)

                if mari_fallback_type:
                    if mari_fallback_type == 'has_hyponym':
                        wn_g_sim -= 0.05
                    elif mari_fallback_type == 'has_hyperonym':
                        wn_g_sim -= 0.10

                if ital_fallback_used:
                    if ital_fallback_type == 'has_hyponym':
                        wn_g_sim -= 0.10
                    elif ital_fallback_type == 'has_hyperonym':
                        wn_g_sim -= 0.10

                total_weighted_similarity, relation_weights, bonus, malus, malus_count, missing_relations, no_gloss_relations, no_gloss_words, total_similarity, bonus_relations = calculate_relation_similarity(
                    wn_g_sim, wm1.get("relations", []), wm2.get("relations", [])
                )

                current_match = {
                    "Literal Lemma": wm1.get("first_literal_lemma", ""),
                    "MariT sense": wm1.get("first_literal_sense", ""),
                    "IWN sense": wm2.get("first_literal_sense", ""),
                    "Gloss S. (WM)": wn_g_sim,
                    "Total S.": total_similarity,
                    "T. Relation S.": total_weighted_similarity,
                    "Mariterm ID": wm1.get("id", ""),
                    "ItalWN ID": wm2.get("id", ""),
                    "Mariterm Gloss": mari_gloss if not fallback_used else f"[FALLBACK] {get_fallback_gloss(wm1, 'near_synonym')[0]}",
                    "ItalWN Gloss": ital_gloss if not ital_fallback_used else f"[FALLBACK] {get_fallback_gloss(wm2, 'near_synonym')[0]}",
                    "MariTerm Relations": format_relations(wm1.get("relations", []), relation_weights),
                    "ItalWN Relations": format_relations(wm2.get("relations", []), relation_weights),
                    "bonus": bonus,
                    "malus": malus,
                    "missing_relations": missing_relations,
                    "no_gloss_relations": no_gloss_relations,
                    "bonus_relations": bonus_relations,
                }

                # Track the best match across all possible pairs for this ItalWN entry
                if wn_g_sim > best_gloss_similarity or \
                   (wn_g_sim == best_gloss_similarity and total_weighted_similarity > best_total_weighted_relation_similarity) or \
                   (wn_g_sim == best_gloss_similarity and total_weighted_similarity == best_total_weighted_relation_similarity and total_similarity > best_total_similarity) or \
                   (wn_g_sim == best_gloss_similarity and total_weighted_similarity == best_total_weighted_relation_similarity and total_similarity == best_total_similarity):

                    best_gloss_similarity = wn_g_sim
                    best_total_weighted_relation_similarity = total_weighted_similarity
                    best_total_similarity = total_similarity
                    best_match = current_match

        # Apply the matching criteria based on thresholds
        if best_match:
            ital_wn_id = best_match.get('ItalWN ID')
            if best_match['Gloss S. (WM)'] >= 0.43 or best_match['T. Relation S.'] > 0:
                best_matches_per_ital_wn[ital_wn_id] = best_match
            elif best_match['Gloss S. (WM)'] >= 0.13 and (best_match['Gloss S. (WM)'] < 0.43 or best_match['T. Relation S.'] > 0):
                best_matches_per_ital_wn[ital_wn_id] = best_match
            elif best_match['T. Relation S.'] > 0.09 or (0.14 <= best_match['Gloss S. (WM)'] < 0.19) or (0.24 <= best_match['Gloss S. (WM)'] < 0.29) or best_match['Gloss S. (WM)'] >= 0.44:
                best_matches_per_ital_wn[ital_wn_id] = best_match

    results = [(match, []) for match in best_matches_per_ital_wn.values()]

    return results

def print_and_return_best_match(results):
    # Sort results by 'Total S.' in descending order
    sorted_results = sorted(results, key=lambda x: x[0].get('Total S.', 0), reverse=True)

    for best_match, matched_senses in sorted_results:
            mari_gloss = best_match.get('Mariterm Gloss', '').strip()
            ital_wn_gloss = best_match.get('ItalWN Gloss', '').strip()

            # Check if both glosses are empty
            if not mari_gloss and not ital_wn_gloss:
                print(f"\nLiteral Lemma: {best_match.get('Literal Lemma', 'N/A')}")
                print(f"MariTerm Gloss: [FALLBACK]")
                print(f"ItalWN Gloss: [FALLBACK]")
                print(f"MariTerm ID: {best_match['Mariterm ID']}")
                print(f"ItalWN ID: {best_match['ItalWN ID']}")
                print(f"MariTerm Sense: {best_match.get('MariT sense', 'N/A')}")
                print(f"ItalWN Sense: {best_match.get('IWN sense', 'N/A')}")
                print("No alternate senses available for comparison.")
                print(f"Gloss S. (WM): 0.00")
                print(f"Total Weighted Relation Similarity: 0.00")
                print(f"  Bonus: 0.00 (0/0 - )")
                print(f"  Malus: 0.00 (0/0 - No Gloss in ItalWN: 0,  - Missing: )")
                print(f"Total S.: 0.00")
                print(f"MariTerm Relations: ")
                print(f"ItalWN Relations: ")
            else:
                # Continue with fallback logic if either gloss is present
                if mari_gloss == '':
                    mari_gloss, mari_fallback_type = get_fallback_gloss(best_match, 'near_synonym')
                    fallback_used = True
                else:
                    mari_fallback_type = None
                    fallback_used = False

                if ital_wn_gloss == '':
                    ital_wn_gloss, ital_fallback_type = get_fallback_gloss(best_match, 'near_synonym')
                else:
                    ital_fallback_type = None

                print(f"\nLiteral Lemma: {best_match.get('Literal Lemma', 'N/A')}")
                print(f"MariTerm Gloss: {mari_gloss}")
                print(f"ItalWN Gloss: {ital_wn_gloss}")
                print(f"MariTerm ID: {best_match['Mariterm ID']}")
                print(f"ItalWN ID: {best_match['ItalWN ID']}")
                print(f"MariTerm Sense: {best_match.get('MariT sense', 'N/A')}")
                print(f"ItalWN Sense: {best_match.get('IWN sense', 'N/A')}")
                print(f"Gloss S. (WM): {best_match['Gloss S. (WM)']:.2f}")
                print(f"Total Weighted Relation Similarity: {best_match['T. Relation S.']:.2f}")
                print(f"  Bonus: {best_match['bonus']:.2f} ({len(best_match['bonus_relations'])}/{len(best_match['MariTerm Relations'])} - {', '.join(best_match['bonus_relations'])})")
                print(f"  Malus: {best_match['malus']:.2f} ({len(best_match['missing_relations']) + len(best_match['no_gloss_relations'])}/{len(best_match['MariTerm Relations'])} - No Gloss in ItalWN: {len(best_match['no_gloss_relations'])}, {', '.join(best_match['no_gloss_relations'])} - Missing: {', '.join(best_match['missing_relations'])})")
                print(f"Total S.: {best_match.get('Total S.', 0.0):.2f}")  # Default to 0.0 if 'Total S.' is not present
                print(f"MariTerm Relations:")
                for rel in best_match.get('MariTerm Relations', []):
                    print(f"   {rel}")
                print(f"ItalWN Relations:")
                for rel in best_match.get('ItalWN Relations', []):
                    print(f"   {rel}")

def score_candidates(MariT, ItalWN, candidates_file):
    mari_term_wms = extract_word_meanings(parse_xml(MariT))
    ital_wn_wms = extract_word_meanings(parse_xml(ItalWN))

    candidate_lemmas = pd.read_csv(candidates_file, delimiter=";")['Shared Lemma'].str.lower().apply(normalize_text).tolist()

    mari_term_wms = [wm for wm in mari_term_wms if any(normalized_lemma in candidate_lemmas for normalized_lemma in wm['normalized_lemmas'])]
    ital_wn_wms = [wm for wm in ital_wn_wms if any(normalized_lemma in candidate_lemmas for normalized_lemma in wm['normalized_lemmas'])]

    results = match_lemmas_with_alternate_senses(mari_term_wms, ital_wn_wms)
    print_and_return_best_match(results)

def save_entries(MariT, ItalWN, breakdown_csv):
    mari_term_wms = extract_word_meanings(parse_xml(MariT))
    ital_wn_wms = extract_word_meanings(parse_xml(ItalWN))

    candidate_lemmas = set(pd.read_csv(breakdown_csv, delimiter=';')['Shared Lemma'].str.lower().apply(normalize_text))

    mari_term_wms = [wm for wm in mari_term_wms if any(normalized_lemma in candidate_lemmas for normalized_lemma in wm['normalized_lemmas'])]
    ital_wn_wms = [wm for wm in ital_wn_wms if any(normalized_lemma in candidate_lemmas for normalized_lemma in wm['normalized_lemmas'])]

    results = match_lemmas_with_alternate_senses(mari_term_wms, ital_wn_wms)

    mari_term_id_to_ital_wn_ids = {}
    ital_wn_id_counts = {}

    for best_match, matched_senses in results:
        mari_term_id = best_match.get('Mariterm ID')
        ital_wn_id = best_match.get('ItalWN ID')

        if mari_term_id not in mari_term_id_to_ital_wn_ids:
            mari_term_id_to_ital_wn_ids[mari_term_id] = [ital_wn_id]
        else:
            mari_term_id_to_ital_wn_ids[mari_term_id].append(ital_wn_id)

        if ital_wn_id in ital_wn_id_counts:
            ital_wn_id_counts[ital_wn_id] += 1
        else:
            ital_wn_id_counts[ital_wn_id] = 1

    unique_entries = []

    for best_match, matched_senses in results:
        mari_term_id = best_match.get('Mariterm ID')
        ital_wn_id = best_match.get('ItalWN ID')

        if len(mari_term_id_to_ital_wn_ids[mari_term_id]) == 1 and ital_wn_id_counts[ital_wn_id] == 1:
            gloss = best_match.get("Mariterm Gloss", 'No Gloss Available')
            if "[FALLBACK]" in gloss:
                gloss = ''  # Remove the gloss entirely if it contains "[FALLBACK]"

            entry = {
                "Literal Lemma": best_match.get("Literal Lemma", 'N/A'),
                "Mariterm ID": mari_term_id,
                "Total S.": best_match.get("Total S.", 0.00),
                "Gloss S. (WM)": best_match.get("Gloss S. (WM)", 0.00),
                "T. Relation S.": best_match.get("T. Relation S.", 0.00),
                "Malus": best_match.get('malus', 0.00),  # Correctly storing malus
                "Bonus": best_match.get('bonus', 0.00),  # Correctly storing bonus
                "Mariterm Gloss": gloss,
                "MariT sense": best_match.get("MariT sense", 'N/A'),
                "IWN sense": best_match.get("IWN sense", 'N/A'),
                "MariTerm Relations": best_match.get("MariTerm Relations", []),
                "ItalWN Relations": best_match.get("ItalWN Relations", []),
            }
            unique_entries.append(entry)

    print(f"Unique Entries: {len(unique_entries)} entries generated.")  # Debugging information

    return unique_entries


unique_entries = save_entries(MariT, ItalWN, candidates_csv)
score_candidates(MariT, ItalWN, candidates_csv)


In [ ]:
def format_results(MariT, ItalWN, breakdown_csv):
    unique_entries = save_entries(MariT, ItalWN, breakdown_csv)

    if not unique_entries:
        print("No entries found. Exiting.")
        return []

    total_similarities = [entry['Total S.'] for entry in unique_entries]
    min_similarity = min(total_similarities)
    max_similarity = max(total_similarities)

    unique_entries.sort(key=lambda x: x['Total S.'], reverse=True)

    formatted_results = []

    for entry in unique_entries:
        literal_lemma = entry.get('Literal Lemma', 'N/A')
        mari_term_id = entry.get('Mariterm ID', 'N/A')
        mari_t_sense = entry.get('MariT sense', 'N/A')
        iwn_sense = entry.get('IWN sense', 'N/A')
        wn_g_sim = entry.get("Gloss S. (WM)", 0.0)
        total_similarity = entry.get('Total S.', 0.00)
        total_weighted_similarity = entry.get('T. Relation S.', 0.00)
        malus = entry.get('Malus', 0.00)  # Ensure the 'Malus' value is retrieved correctly
        bonus = entry.get('Bonus', 0.00)  # Ensure the 'Bonus' value is retrieved correctly
        gloss = entry.get('Mariterm Gloss', 'No Gloss Available')

        formatted_results.append({
            'Literal Lemma': literal_lemma,
            'MariTerm ID': mari_term_id,
            'MariT sense': mari_t_sense,
            'IWN sense': iwn_sense,
            'Gloss S. (WM)': wn_g_sim,
            'Total S.': total_similarity,
            'T. Relation S.': total_weighted_similarity,
            'Malus': malus,  # Correctly reference 'Malus' key
            'Bonus': bonus,  # Correctly reference 'Bonus' key
            'MariTerm Gloss': gloss,
        })

    return formatted_results

def print_formatted_results(formatted_results):
    if not formatted_results:
        print("No valid entries to display.")
        return

    print(f"{'Literal Lemma':<25} {'MariTerm ID':<15} {'MariT sense':<12} {'IWN sense':<12} {'Gloss S. (WM)':<15} {'Total S.':<15} {'T. Relation S.':<20} {'Malus':<10} {'Bonus':<10} {'MariTerm Gloss':<50}")
    print('-' * 250)

    for result in formatted_results:
        print(f"{result['Literal Lemma']:<25} {result['MariTerm ID']:<15} {result['MariT sense']:<12} {result['IWN sense']:<12} "
              f"{result['Gloss S. (WM)']:<15.2f} {result['Total S.']:<15.2f} {result['T. Relation S.']:<20.2f}"
              f"{result['Malus']:<10.2f} {result['Bonus']:<10.2f} {result['MariTerm Gloss']:<50}")

def store_to_csv(formatted_results, breakdown_csv):
    # Entries with "[FALLBACK]" glosses have already had the gloss removed during the formatting step
    df = pd.DataFrame(formatted_results)

    if 'Gloss S. (WM)' in df.columns:
        df['Gloss S. (WM)'] = pd.to_numeric(df['Gloss S. (WM)'], errors='coerce')
    else:
        print("Warning: 'Gloss S. (WM)' column is missing!")

    df['Malus'] = pd.to_numeric(df['Malus'], errors='coerce')
    df['Bonus'] = pd.to_numeric(df['Bonus'], errors='coerce')

    required_columns = ['Literal Lemma', 'MariTerm ID', 'MariT sense', 'IWN sense', 'Gloss S. (WM)', 'Total S.', 'T. Relation S.', 'Malus', 'Bonus', 'MariTerm Gloss']
    for col in required_columns:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    df.to_csv(breakdown_csv, sep=';', encoding='utf-8', index=False, float_format='%.2f')


# Main Execution
formatted_results = format_results(MariT, ItalWN, candidates_csv)
if formatted_results:
    print_formatted_results(formatted_results)
    store_to_csv(formatted_results, breakdown_csv)
else:
    print("No results were generated.")


In [ ]:
# Function to validate XML
def validate_xml(file_path):
    try:
        ET.parse(file_path)
        return True
    except ET.ParseError as e:
        print(f"Error parsing {file_path}: {e}")
        return False

# Filter meanings based on lemmas and senses
def filter_wm(xml_root):
    word_meanings = []
    for word_meaning in xml_root.findall(".//WORD_MEANING"):
        lemma_variants = word_meaning.find("VARIANTS")
        lemmas = [literal.get("LEMMA") for literal in lemma_variants.findall("LITERAL") if literal.get("LEMMA")]
        senses = [literal.get("SENSE") for literal in lemma_variants.findall("LITERAL") if literal.get("SENSE")]
        first_literal = lemma_variants.find("LITERAL")
        first_literal_lemma = first_literal.get("LEMMA") if first_literal is not None else "No Literal Lemma"
        first_literal_sense = first_literal.get("SENSE") if first_literal is not None else "No Sense"

        word_meanings.append({
            "id": word_meaning.get("ID"),
            "lemmas": lemmas,
            "senses": senses,
            "first_literal_lemma": first_literal_lemma,
            "first_literal_sense": first_literal_sense,
            "original_element": word_meaning
        })

    return word_meanings

# Match lemmas based on first literal lemma and sense numbers from the CSV
def match_lemmas(mari_term_wms, ital_wn_wms, csv_entries):
    matched_pairs = []
    for entry in csv_entries:
        csv_lemma = entry['literal_lemma'].lower()
        mari_t_sense = entry['mari_t_sense']
        iwn_sense = entry['iwn_sense']

        # Find the correct word meanings based on both the lemma and the sense numbers
        mari_match = next((wm for wm in mari_term_wms if wm["first_literal_lemma"].lower() == csv_lemma and wm["first_literal_sense"] == mari_t_sense), None)
        iwn_match = next((wm for wm in ital_wn_wms if wm["first_literal_lemma"].lower() == csv_lemma and wm["first_literal_sense"] == iwn_sense), None)

        if mari_match and iwn_match:
            matched_pairs.append((mari_match["original_element"], iwn_match["original_element"]))
        else:
            print(f"Skipped: Could not find matching sense for {csv_lemma} (MariTerm sense {mari_t_sense}, IWN sense {iwn_sense})")

    return matched_pairs

# Transcribe matched pairs to new XML files
def transcribe_matched_pairs(matched_pairs, filt_mart, filt_iwn):
    mari_term_root = ET.Element("WN")
    ital_wn_root = ET.Element("WN")

    for mari_element, ital_element in matched_pairs:
        mari_term_root.append(mari_element)
        ital_wn_root.append(ital_element)

    mari_term_tree = ET.ElementTree(mari_term_root)
    mari_term_tree.write(filt_mart, encoding="utf-8", xml_declaration=True)

    ital_wn_tree = ET.ElementTree(ital_wn_root)
    ital_wn_tree.write(filt_iwn, encoding="utf-8", xml_declaration=True)

# Main function to process and transcribe candidate entries
def transcribe_candidates(MariT, ItalWN, candidates_file, filt_mart, filt_iwn):
    if not validate_xml(MariT) or not validate_xml(ItalWN):
        return

    mari_term_wms = filter_wm(ET.parse(MariT).getroot())
    ital_wn_wms = filter_wm(ET.parse(ItalWN).getroot())

    # Read the CSV file to extract the lemmas and their corresponding senses
    csv_entries = []
    with open(candidates_file, mode='r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=';')
        for row in reader:
            csv_entries.append({
                'literal_lemma': row['Literal Lemma'],
                'mari_t_sense': row.get('MariT sense', 'N/A'),
                'iwn_sense': row.get('IWN sense', 'N/A')
            })

    matched_pairs = match_lemmas(mari_term_wms, ital_wn_wms, csv_entries)
    transcribe_matched_pairs(matched_pairs, filt_mart, filt_iwn)
    print(f"Total matched pairs transcribed: {len(matched_pairs)}")

# Process and transcribe candidates
transcribe_candidates(MariT, ItalWN, breakdown_csv, filt_mart, filt_iwn)


# Part 2
Updating synsets

In [ ]:
replaced_glosses = []
new_word_meanings = []

def read_lemmas_from_csv(file_path):
    lemmas = []
    with open(file_path, mode='r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=';')
        for row in reader:
            gloss_similarity = float(row['Gloss S. (WM)'])
            total_weighted_similarity = float(row['T. Relation S.'])
            total_similarity = float(row['Total S.'])
            malus = float(row['Malus'])
            literal_lemma = row['Literal Lemma']
            mari_t_sense = row.get('MariT sense', 'N/A')
            iwn_sense = row.get('IWN sense', 'N/A')

            if gloss_similarity >= 0.43 or total_weighted_similarity > 0.00:
                lemmas.append({'literal_lemma': literal_lemma, 'mari_t_sense': mari_t_sense, 'iwn_sense': iwn_sense})
            elif gloss_similarity == 0 or malus == 0:
                print(f"Skipped - G.Sim (WM) or Malus = 0: {literal_lemma} | MariTerm Sense: {mari_t_sense} | ItalWN Sense: {iwn_sense}")
            else:
                if mari_t_sense == iwn_sense:
                    if (
                        (0.29 <= gloss_similarity < 0.42 and (-0.03 <= total_similarity or total_similarity <= -0.35)) or
                        (0.16 < gloss_similarity < 0.29 and (malus == -0.33 or total_similarity >= -0.42)) or
                        (0.13 <= gloss_similarity <= 0.16 and (-0.83 <= total_similarity <= -0.20)) or
                        (0.10 <= gloss_similarity <= 0.12 and total_similarity >= -0.55) or
                        (0.06 <= gloss_similarity <= 0.09 and (-0.60 <= total_similarity <= -0.26)) or
                        (0 < gloss_similarity <= 0.05 and (-0.61 <= total_similarity <= -0.28))
                    ):
                        lemmas.append({'literal_lemma': literal_lemma, 'mari_t_sense': mari_t_sense, 'iwn_sense': iwn_sense})
                else:
                    if (
                        (0.29 < gloss_similarity < 0.42 and total_similarity >= -0.02) or
                        (0.25 <= gloss_similarity < 0.28 and total_similarity >= -1.07) or
                        (0.19 <= gloss_similarity <= 0.24 and (-4.40 < total_similarity < -0.11 or total_similarity >= -0.76)) or
                        (0.16 <= gloss_similarity <= 0.18 and total_similarity >= -1.14) or
                        (0.13 <= gloss_similarity <= 0.15 and -1.52 <= total_similarity < -0.18) or
                        (0.10 <= gloss_similarity < 0.12 and (total_similarity >= -0.55 or total_similarity == -0.89)) or
                        (0.07 < gloss_similarity <= 0.09 and (-2.55 <= total_similarity <= -0.25 or total_similarity == -2.55)) or
                        (0 < gloss_similarity <= 0.06 and total_similarity < -0.63)
                    ):
                        lemmas.append({'literal_lemma': literal_lemma, 'mari_t_sense': mari_t_sense, 'iwn_sense': iwn_sense})
                    else:
                        print(f"Skipped - other : {literal_lemma} | MariTerm Sense: {mari_t_sense} | ItalWN Sense: {iwn_sense}")

    print(f"Final Selected Lemmas: {lemmas}")
    print(f"Total selected:{len(lemmas)}")
    return lemmas

def retrieve_wm(root):
    return {(word.find('.//LITERAL').get('LEMMA'), word.find('.//LITERAL').get('SENSE')): word for word in root.findall('.//WORD_MEANING')}

def find_word_meaning_in_root(root, lemma, sense):
    """
    Utility function to find a word meaning in a given XML root by lemma and sense.
    """
    for wm in root.findall('.//WORD_MEANING'):
        for literal in wm.findall('.//LITERAL'):
            if literal.get('LEMMA') == lemma and literal.get('SENSE') == sense:
                return wm
    return None

def get_new_id(existing_ids):
    numeric_ids = [int(id.split('#')[1]) for id in existing_ids if '#' in id and id.split('#')[1].isdigit()]
    return max(numeric_ids, default=0) + 1

def get_gloss_from_italwn(lemma, sense, root, pos=None):
    for word in root.findall('.//WORD_MEANING'):
        literal = word.find('.//LITERAL')
        word_pos = word.get('PART_OF_SPEECH')
        if literal is not None and literal.get('LEMMA') == lemma and literal.get('SENSE') == sense and (pos is None or word_pos == pos):
            gloss_element = word.find('.//GLOSS')
            if gloss_element is not None and gloss_element.text:
                return gloss_element.text.strip()
    return None

def extract_target_ids(root):
    return {(word.find('.//LITERAL').get('LEMMA'), word.find('.//LITERAL').get('SENSE')): word.get('ID') for word in root.findall('.//WORD_MEANING')}

def retrieve_relation_ids(iwn_root):
    return {
        rel.get('TYPE'): {'ID': rel.get('ID'), 'INV_ID': rel.get('INV_ID')}
        for rel in iwn_root.findall('.//RELATION') if rel.get('ID') and rel.get('INV_ID')
    }

def determine_inverse_relation(rel_type):
    inverse_map = {
        "has_hyperonym": "has_hyponym",
        "has_hyponym": "has_hyperonym",
        "near_synonym": "near_synonym",
        "has_xpos_hyperonym": "has_xpos_hyponym",
        "has_xpos_hyponym": "has_xpos_hyperonym",
        "xpos_near_synonym": "xpos_near_synonym"
    }
    return inverse_map.get(rel_type, rel_type)

def clean_target_wm_id(target_wm):
    """Remove any prefix (e.g., 'V#') from the ID of a TARGET_WM element."""
    clean_id = target_wm.get('ID').split('#')[-1]
    target_wm.set('ID', clean_id)

def remove_duplicate_relations(iwn_root):
    """
    Remove duplicate relations within each WORD_MEANING in the XML.
    """
    for word_meaning in iwn_root.findall('.//WORD_MEANING'):
        internal_links = word_meaning.find('.//INTERNAL_LINKS')
        if internal_links is not None:
            # Use a set to track unique relation signatures
            seen_relations = set()
            relations_to_remove = []

            for relation in internal_links.findall('.//RELATION'):
                # Create a unique identifier for each relation based on its type, target WM ID, and sense
                relation_signature = (
                    relation.get('TYPE'),
                    relation.find('.//TARGET_WM').get('ID'),
                    relation.find('.//TARGET_WM').get('SENSE')
                )

                # If this signature has already been seen, mark the relation for removal
                if relation_signature in seen_relations:
                    relations_to_remove.append(relation)
                else:
                    seen_relations.add(relation_signature)

            # Remove the duplicates
            for relation in relations_to_remove:
                internal_links.remove(relation)

def extract_eq_links_and_top_onto(root):
    eq_links, top_onto = {}, {}
    for word in root.findall('.//WORD_MEANING'):
        key = (word.find('.//LITERAL').get('LEMMA'), word.find('.//LITERAL').get('SENSE'))

        # Explicitly check if the elements exist
        eq_link_elem = word.find('.//EQ_LINKS')
        top_onto_elem = word.find('.//TOP_ONTO')

        eq_links[key] = eq_link_elem if eq_link_elem is not None else ET.Element('EQ_LINKS')
        top_onto[key] = top_onto_elem if top_onto_elem is not None else ET.Element('TOP_ONTO')

    return eq_links, top_onto

def merge_sections(existing_section, new_section):
    existing_relations = {ET.tostring(rel) for rel in existing_section.findall('.//RELATION')}
    for new_rel in new_section.findall('.//RELATION'):
        if ET.tostring(new_rel) not in existing_relations:
            # Clean the IDs in the TARGET_WM elements
            for target_wm in new_rel.findall('.//TARGET_WM'):
                clean_target_wm_id(target_wm)
            existing_section.append(new_rel)

def clean_existing_target_wm_ids(root):
    """Clean the IDs of all TARGET_WM elements in the provided root."""
    for target_wm in root.findall('.//TARGET_WM'):
        clean_target_wm_id(target_wm)

def clean_and_move_eq_links_at_end(iwn_root):
    """
    This function ensures that any 'eq_*' relations that ended up in INTERNAL_LINKS
    are moved to the EQ_LINKS section, if applicable.
    It also sorts the relations first by ID and then alphabetically by lemma.
    """
    for word_meaning in iwn_root.findall('.//WORD_MEANING'):
        internal_links = word_meaning.find('.//INTERNAL_LINKS')
        eq_links_section = word_meaning.find('.//EQ_LINKS')

        # Updated line to avoid DeprecationWarning
        if internal_links is not None and eq_links_section is not None:
            # Collect any eq_* relations
            eq_relations_to_move = [rel for rel in internal_links.findall('.//RELATION') if rel.get('TYPE', '').startswith('eq_')]

            # Move them to EQ_LINKS
            for rel in eq_relations_to_move:
                internal_links.remove(rel)
                eq_links_section.append(rel)

        # Updated line to avoid DeprecationWarning
        if internal_links is not None:
            relations = sorted(
                internal_links.findall('.//RELATION'),
                key=lambda r: (
                    int(r.get('ID')) if r.get('ID', '').isdigit() else float('inf'),
                    r.find('.//TARGET_WM').get('LEMMA', '').lower()
                )
            )

            # Clear existing relations and re-append sorted relations
            for rel in internal_links.findall('.//RELATION'):
                internal_links.remove(rel)
            for rel in relations:
                internal_links.append(rel)

        # Updated line to avoid DeprecationWarning
        if eq_links_section is not None:
            eq_relations = sorted(
                eq_links_section.findall('.//RELATION'),
                key=lambda r: (
                    int(r.get('ID')) if r.get('ID', '').isdigit() else float('inf'),
                    r.find('.//TARGET_WM').get('LEMMA', '').lower()
                )
            )

            # Clear existing relations and re-append sorted relations
            for rel in eq_links_section.findall('.//RELATION'):
                eq_links_section.remove(rel)
            for rel in eq_relations:
                eq_links_section.append(rel)

def sort_word_meanings_by_lemma_and_sense(root):
    word_meanings = sorted(root.findall('.//WORD_MEANING'), key=lambda wm: (wm.find('.//LITERAL').get('LEMMA').lower(), wm.find('.//LITERAL').get('SENSE')))
    for word in word_meanings:
        root.remove(word)
    for word in word_meanings:
        root.append(word)

def remove_redundant_word_meanings(iwn_root):
    for replacement in replaced_glosses:
        for word_meaning in iwn_root.findall('.//WORD_MEANING'):
            literal = word_meaning.find('.//LITERAL')
            gloss_element = word_meaning.find('.//GLOSS')
            if literal is not None and gloss_element is not None:
                if literal.get('LEMMA') == replacement['lemma'] and gloss_element.text.strip() == replacement['original_gloss']:
                    iwn_root.remove(word_meaning)
                    print(f"Removed redundant word meaning for {replacement['lemma']} with replaced gloss: '{replacement['original_gloss']}'")

def replace_gloss_with_original(iwn_root, original_iwn_root, updated_entries):
    replaced_glosses_set = set()

    for iwn_wm in iwn_root.findall('.//WORD_MEANING'):
        lemma = iwn_wm.find('.//LITERAL').get('LEMMA')
        sense = iwn_wm.find('.//LITERAL').get('SENSE')
        pos = iwn_wm.get('PART_OF_SPEECH')

        if (lemma, sense) in updated_entries:
            continue  # Skip this entry if it has already been updated

        original_gloss = get_gloss_from_italwn(lemma, sense, original_iwn_root, pos)

        if original_gloss:
            iwn_gloss_element = iwn_wm.find('.//GLOSS')
            current_gloss = iwn_gloss_element.text.strip() if iwn_gloss_element is not None and iwn_gloss_element.text else None
            if current_gloss != original_gloss:
                if iwn_gloss_element is not None:
                    iwn_gloss_element.text = original_gloss
                else:
                    iwn_gloss_element = ET.SubElement(iwn_wm, 'GLOSS')
                    iwn_gloss_element.text = original_gloss

                # Print the replacement message only if it hasn't been printed for this lemma-sense-pos combination
                if (lemma, sense, pos) not in replaced_glosses_set:
                    print(f"Replaced WORD_MEANING gloss for '{lemma}' sense '{sense}' with original IWN gloss.")
                    replaced_glosses_set.add((lemma, sense, pos))

            # Replace glosses in TARGET_WM elements
            for target_wm in iwn_root.findall(f'.//TARGET_WM[@LEMMA="{lemma}"][@SENSE="{sense}"][@PART_OF_SPEECH="{pos}"]'):
                target_gloss = target_wm.get('GLOSS')
                if target_gloss != original_gloss:
                    target_wm.set('GLOSS', original_gloss)

                    # Print the replacement message only if it hasn't been printed for this lemma-sense-pos combination
                    if (lemma, sense, pos) not in replaced_glosses_set:
                        print(f"Replaced TARGET_WM gloss for '{lemma}' sense '{sense}' with original IWN gloss.")
                        replaced_glosses_set.add((lemma, sense, pos))

def create_or_merge_word_meaning(lemma, sense, existing_ids, mari_eq_links, mari_top_onto, iwn_root, back_references=[], relation_ids=None, gloss="", add_lfeatures_due_to_hyponym=False, add_lfeatures_due_to_non_compliance=False, original_iwn_root=None):
    updated_entries.add((lemma, sense))
    word_meaning = find_word_meaning_in_root(iwn_root, lemma, sense)
    if word_meaning is None:
        word_meaning = find_word_meaning_in_root(original_iwn_root, lemma, sense)

    if word_meaning is not None:
        word_meaning_id = word_meaning.get('ID')
        gloss_element = word_meaning.find('.//GLOSS')
        original_gloss = gloss_element.text.strip() if gloss_element is not None and gloss_element.text else ''

        if original_gloss != gloss:
            if gloss_element is None:
                gloss_element = ET.SubElement(word_meaning, 'GLOSS')
            gloss_element.text = gloss
        return word_meaning, word_meaning_id

    pos_prefix = back_references[0]['part_of_speech'] if back_references else "N"
    new_id = get_new_id(existing_ids)
    word_meaning_id = f"{pos_prefix}#{new_id}"
    existing_ids.add(word_meaning_id)

    word_meaning = ET.Element('WORD_MEANING', ID=word_meaning_id, PART_OF_SPEECH=pos_prefix)
    gloss_element = ET.SubElement(word_meaning, 'GLOSS')
    gloss_element.text = gloss

    variants = ET.SubElement(word_meaning, 'VARIANTS')
    literal_node = ET.SubElement(variants, 'LITERAL', LEMMA=lemma, SENSE=str(sense))

    if add_lfeatures_due_to_hyponym or add_lfeatures_due_to_non_compliance:
        ET.SubElement(literal_node, 'LFEATURES', SUBLANGUAGE="mar")

    internal_links = ET.SubElement(word_meaning, 'INTERNAL_LINKS')
    eq_links_section = ET.SubElement(word_meaning, 'EQ_LINKS')
    top_onto_section = ET.SubElement(word_meaning, 'TOP_ONTO')

    iwn_root.append(word_meaning)
    new_word_meanings.append(lemma)

    key = (lemma, sense)
    if key in mari_eq_links:
        merge_sections(eq_links_section, mari_eq_links[key])
    if key in mari_top_onto:
        merge_sections(top_onto_section, mari_top_onto[key])

    for ref in back_references:
        rel_type, target_lemma = ref['inverse_relation'], ref['lemma']
        section = internal_links if not rel_type.startswith('eq_') else eq_links_section
        existing_relations = {
            (rel.get('TYPE'), rel.find('.//TARGET_WM').get('LEMMA')): rel
            for rel in section.findall('.//RELATION')
        }

        if (rel_type, target_lemma) not in existing_relations:
            rel_id, inv_id = relation_ids.get(rel_type, {}).get('ID', 'UNKNOWN'), relation_ids.get(rel_type, {}).get('INV_ID', 'UNKNOWN')
            new_rel = ET.SubElement(section, 'RELATION', TYPE=rel_type, ID=rel_id, INV_ID=inv_id)

            target_wm = ET.SubElement(new_rel, 'TARGET_WM', ID=ref['id'], PART_OF_SPEECH=ref['part_of_speech'], LEMMA=ref['lemma'], SENSE=ref['sense'])
            clean_target_wm_id(target_wm)  # Ensure the ID is cleaned

            if ref.get('gloss'):
                target_wm.set('GLOSS', ref['gloss'])

    return word_meaning, word_meaning_id

def update_iwn_entries(selected_lemmas, mari_root, iwn_root, original_mari_root, original_iwn_root):
    mari_meanings = retrieve_wm(mari_root)
    original_target_ids = extract_target_ids(original_iwn_root)
    mari_eq_links, mari_top_onto = extract_eq_links_and_top_onto(original_mari_root)
    existing_ids = {word.get('ID') for word in iwn_root.findall('.//WORD_MEANING')}
    relation_ids = retrieve_relation_ids(iwn_root)

    relations_added = 0
    word_meanings_created = 0
    lemmas_processed = 0

    def retrieve_or_generate_gloss(target_lemma, target_sense, main_gloss=None):
        italwn_gloss = get_gloss_from_italwn(target_lemma, target_sense, iwn_root)
        mariterm_gloss = get_gloss_from_relation_in_mariterm(target_lemma, target_sense, mari_meanings)
        italwn_gloss_original = None

        if not italwn_gloss and not mariterm_gloss:
            italwn_gloss_original = get_gloss_from_italwn(target_lemma, target_sense, original_iwn_root)
            if italwn_gloss_original:
                print(f"Special case: Found gloss for '{target_lemma}' sense '{target_sense}' in original IWN.")
                return italwn_gloss_original

        if mariterm_gloss and italwn_gloss and mariterm_gloss != italwn_gloss:
            replaced_glosses.append({"lemma": target_lemma, "sense": target_sense, "original_gloss": mariterm_gloss})
            print(f"Replacing gloss for '{target_lemma}' (sense {target_sense}). Original: '{mariterm_gloss}', New: '{italwn_gloss}'")

        return italwn_gloss if italwn_gloss else mariterm_gloss

    def get_gloss_from_relation_in_mariterm(target_lemma, target_sense, mari_meanings):
        for mari_word in mari_meanings.values():
            for rel in mari_word.findall('.//INTERNAL_LINKS//RELATION'):
                target_wm = rel.find('.//TARGET_WM')
                if target_wm is not None and target_wm.get('LEMMA') == target_lemma and target_wm.get('SENSE') == target_sense:
                    return target_wm.get('GLOSS', '').strip()
        return ""

    for entry in selected_lemmas:
        lemma, mari_sense, iwn_sense = entry['literal_lemma'], entry['mari_t_sense'], entry['iwn_sense']
        lemmas_processed += 1

        mari_word = mari_meanings.get((lemma, mari_sense))
        if mari_word is None:
            print(f"Skipped: MariTerm sense {mari_sense} for {lemma} not found in MariTerm filtered.")
            continue  # Skip to the next iteration if mari_word is None

        mari_gloss_element = mari_word.find('.//GLOSS')
        mari_gloss = mari_gloss_element.text.strip() if mari_gloss_element is not None and mari_gloss_element.text else ''

        if any(r['lemma'] == lemma and r['sense'] == mari_sense and r['original_gloss'] == mari_gloss for r in replaced_glosses):
            print(f"Skipping redundant creation for lemma '{lemma}' with sense '{mari_sense}' and gloss '{mari_gloss}'")
            continue

        word_meaning, _ = create_or_merge_word_meaning(
            lemma, iwn_sense, existing_ids, mari_eq_links, mari_top_onto, iwn_root, [], relation_ids,
            gloss=retrieve_or_generate_gloss(lemma, iwn_sense, mari_gloss),
            original_iwn_root=original_iwn_root
        )
        word_meanings_created += 1 if word_meaning is not None else 0

        for rel in mari_word.findall('.//INTERNAL_LINKS//RELATION'):
            rel_type = rel.get('TYPE')
            if rel_type not in allowed_relation_types:
                continue

            target_lemma = rel.find('.//TARGET_WM').get('LEMMA', '')
            target_sense = rel.find('.//TARGET_WM').get('SENSE')
            target_gloss = retrieve_or_generate_gloss(target_lemma, target_sense, mari_gloss)

            target_id = original_target_ids.get((target_lemma, target_sense))
            if not target_id:
                back_references = [{
                    "id": word_meaning.get('ID'),
                    "inv_id": relation_ids[rel_type]['INV_ID'],
                    "part_of_speech": word_meaning.get('PART_OF_SPEECH'),
                    "lemma": lemma,
                    "sense": iwn_sense,
                    "gloss": mari_gloss,
                    "inverse_relation": determine_inverse_relation(rel_type)
                }]
                new_wm, target_id = create_or_merge_word_meaning(
                    target_lemma, target_sense, existing_ids, mari_eq_links, mari_top_onto, iwn_root, back_references, relation_ids,
                    original_iwn_root=original_iwn_root
                )
                word_meanings_created += 1 if new_wm is not None else 0

            if rel_type in relation_ids:
                new_rel = ET.SubElement(word_meaning.find('.//INTERNAL_LINKS'), 'RELATION', TYPE=rel_type, ID=relation_ids[rel_type]['ID'], INV_ID=relation_ids[rel_type]['INV_ID'])
                new_target_wm = ET.SubElement(new_rel, 'TARGET_WM', ID=target_id.replace(f"{word_meaning.get('PART_OF_SPEECH')}#", ""), PART_OF_SPEECH=rel.find('.//TARGET_WM').get('PART_OF_SPEECH'), LEMMA=target_lemma, SENSE=target_sense)
                if target_gloss:
                    new_target_wm.set('GLOSS', target_gloss)
                relations_added += 1

    clean_and_move_eq_links_at_end(iwn_root)
    remove_redundant_word_meanings(iwn_root)
    sort_word_meanings_by_lemma_and_sense(iwn_root)
    replace_gloss_with_original(iwn_root, original_iwn_root, updated_entries)

    #print(f"Total lemmas processed: {lemmas_processed}")
    # Print the total number of word meanings in the final IWN file
    total_word_meanings = len(iwn_root.findall('.//WORD_MEANING'))
    print(f"Total relations added: {relations_added}")
    print(f"New Word Meanings Created: {new_word_meanings}")
    print(f"Total brand new WM: {len(new_word_meanings)}")
    print(f"Total word meanings in the file: {total_word_meanings}")

# Process CSV and XML files
selected_lemmas = read_lemmas_from_csv(breakdown_csv)
tree_mari = ET.parse(filt_mart)
mari_root = tree_mari.getroot()
tree_iwn = ET.parse(filt_iwn)
iwn_root = tree_iwn.getroot()
tree_original_mari = ET.parse(MariT)
original_mari_root = tree_original_mari.getroot()
tree_original_iwn = ET.parse(ItalWN)
original_iwn_root = tree_original_iwn.getroot()


updated_entries = set()  # Track updated (lemma, sense) pairs

# Update IWN entries
update_iwn_entries(selected_lemmas, mari_root, iwn_root, original_mari_root, original_iwn_root)

replace_gloss_with_original(iwn_root, original_iwn_root, updated_entries)

# Clean up TARGET_WM IDs to ensure they only contain numbers
clean_existing_target_wm_ids(iwn_root)

# Remove duplicate relations
remove_duplicate_relations(iwn_root)

# Save the updated IWN file
ET.ElementTree(iwn_root).write(updates, encoding='utf-8', xml_declaration=True)
print("Update complete.")

In [ ]:
def merge_and_format_iwn_files(ItalWN, updates, IWN_pre_mod):
    # Parse the original unfiltered IWN and the final output files
    tree_original = ET.parse(ItalWN)
    root_original = tree_original.getroot()

    tree_output = ET.parse(updates)
    root_output = tree_output.getroot()

    # Create a dictionary to store word meanings from the final output file by (lemma, sense)
    output_wm_dict = {}
    for word_meaning in root_output.findall('.//WORD_MEANING'):
        lemma = word_meaning.find('.//LITERAL').get('LEMMA')
        sense = word_meaning.find('.//LITERAL').get('SENSE')
        output_wm_dict[(lemma, sense)] = word_meaning

    # Merge the original IWN with the final output, replacing old entries with new ones
    merged_wm_list = []
    for word_meaning in root_original.findall('.//WORD_MEANING'):
        lemma = word_meaning.find('.//LITERAL').get('LEMMA')
        sense = word_meaning.find('.//LITERAL').get('SENSE')

        # If there's an updated version in the output file, use that one
        if (lemma, sense) in output_wm_dict:
            merged_wm_list.append(output_wm_dict.pop((lemma, sense)))
        else:
            merged_wm_list.append(word_meaning)

    # Add any remaining new word meanings from the final output file
    merged_wm_list.extend(output_wm_dict.values())

    # Sort the word meanings alphabetically by the first LITERAL LEMMA
    merged_wm_list.sort(key=lambda wm: (wm.find('.//LITERAL').get('LEMMA').lower(), wm.find('.//LITERAL').get('SENSE')))

    # Create a new XML tree for the merged content
    new_root = ET.Element(root_original.tag)

    for word_meaning in merged_wm_list:
        new_root.append(word_meaning)

    # Write the merged and formatted XML to a new file
    tree_merged = ET.ElementTree(new_root)
    tree_merged.write(IWN_pre_mod, encoding='utf-8', xml_declaration=True)

    # Format the XML with indentation
    format_xml_with_indentation(IWN_pre_mod)

    print(f"Merged IWN file created and saved as '{IWN_pre_mod}'")

def format_xml_with_indentation(file_path):
    """Format the XML file with proper indentation and remove unnecessary blank lines."""
    # Read the file and parse it into an XML document
    with open(file_path, 'r', encoding='utf-8') as file:
        xml_str = file.read()

    # Parse the XML string
    dom = xml.dom.minidom.parseString(xml_str)

    # Pretty print with indentation
    pretty_xml_as_string = dom.toprettyxml(indent="  ")

    # Remove extra blank lines
    pretty_xml_as_string = re.sub(r'\n\s*\n', '\n', pretty_xml_as_string)

    # Write the cleaned and pretty-printed XML back to the file
    with open(file_path, 'w', encoding='utf-8') as file:
        file.write(pretty_xml_as_string)

merge_and_format_iwn_files(ItalWN, updates, IWN_pre_mod)


# [Post-hoc]
A. identification of different synset with overlapping id

B. Extraction of the number of WORD MEANING @IDs to be entered for the manual creation of new entries in IWN or the correction of existing ones

C. Tracing of new entries and added semantic relationships

D. Tracking and resolving inconsistencies in glosses (glosses missing in the OG synset but contained in a corresponding target synset)

E. Adding a_LINKS PLUG-IN to all entries, with due information on the associated MariTerm entry (where applicable)

F. Reversing process in _§E_ for MariTerm with IWN entries

In [ ]:
def extract_ids_and_lemmas(xml_file):
    """Extracts IDs and LEMMA values from the XML file."""
    data = {}
    
    # Parse the XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()
    
    # Iterate through each WORD_MEANING element
    for word_meaning in root.findall('.//WORD_MEANING'):
        # Extract the numerical ID and LEMMA
        word_id = word_meaning.get('ID').split('#')[1]  # Extract the numerical part
        lemma = word_meaning.find('.//VARIANTS/LITERAL').get('LEMMA')
        
        # Store in dictionary with ID as key and lemma as value (store as a list for handling multiple lemmas)
        if word_id not in data:
            data[word_id] = []
        data[word_id].append(lemma)
    
    return data

def compare_files(file1_data, file2_data):
    """Compares IDs and LEMMAs between two files and prints pairs where ID matches but lemmas differ."""
    # Combine data from both files
    all_data = {**file1_data, **file2_data}
    
    # Check for matching IDs with different LEMMAs
    for word_id, lemmas in all_data.items():
        if len(lemmas) > 1:  # Only consider IDs with more than one lemma
            lemmas_set = set(lemmas)  # Use set to remove duplicates
            if len(lemmas_set) > 1:  # If there are different lemmas for the same ID
                print(f"{word_id}, {lemmas[0]} | {word_id}, {', '.join(lemmas_set - {lemmas[0]})} ")

def main(file1, file2):
    # Extract data from both files
    file1_data = extract_ids_and_lemmas(ItalWN)
    file2_data = extract_ids_and_lemmas(IWN_pre_mod)
    
    # Compare the data and print the results
    compare_files(file1_data, file2_data)

# Example usage

main(ItalWN, IWN_pre_mod)

In [ ]:
def get_last_word_meaning_id(file):
    """
    This function reads an XML file and returns the highest numeric ID of a Word Meaning (WM).
    """
    tree = ET.parse(IWN_post_mm)
    root = tree.getroot()

    # Initialize the highest ID variable
    max_id = 0

    # Iterate through all WORD_MEANING elements
    for word_meaning in root.findall('.//WORD_MEANING'):
        wm_id = word_meaning.get('ID')
        if wm_id and '#' in wm_id:
            # Extract the numeric part of the ID
            numeric_id = int(wm_id.split('#')[-1])
            # Update max_id if the current one is higher
            if numeric_id > max_id:
                max_id = numeric_id

    return max_id + 1

# Example usage:
last_id = get_last_word_meaning_id(IWN_post_mm)
print(f"Next ID to add: {last_id}")


In [ ]:
updated_synsets = []
new_synsets = []
new_word_meanings_info = []
newly_added_relations_info = []


def extract_internal_links(wm):
    """Extract all relations in INTERNAL_LINKS."""
    return {(rel.get('TYPE'), rel.find('.//TARGET_WM').get('LEMMA'), rel.find('.//TARGET_WM').get('SENSE'), rel.find('.//TARGET_WM').get('ID')): rel
            for rel in wm.findall('.//INTERNAL_LINKS//RELATION')}

def identify_updates_in_iwn(ItalWN, plug_att):
    # Parsing the XML files
    original_tree = ET.parse(ItalWN)
    original_root = original_tree.getroot()

    finalized_tree = ET.parse(plug_att)
    finalized_root = finalized_tree.getroot()

    # Dictionary of original word meanings
    original_word_meanings = {
        (wm.find('.//LITERAL').get('LEMMA'), wm.find('.//LITERAL').get('SENSE')): wm
        for wm in original_root.findall('.//WORD_MEANING')
    }

    # Iterating through finalized word meanings
    for finalized_wm in finalized_root.findall('.//WORD_MEANING'):
        lemma = finalized_wm.find('.//LITERAL').get('LEMMA')
        sense = finalized_wm.find('.//LITERAL').get('SENSE')
        synset_id = finalized_wm.get('ID')  # Get the ID of the current synset

        if (lemma, sense) not in original_word_meanings:
            new_synsets.append({
            'LEMMA': lemma,
            'SENSE': sense,
            'ID': synset_id,
            })
            new_word_meanings_info.append((lemma, sense))
        else:
            original_wm = original_word_meanings[(lemma, sense)]

            original_internal_links = extract_internal_links(original_wm)
            finalized_internal_links = extract_internal_links(finalized_wm)

            # Identify new relations that are present in the finalized version but not in the original
            new_relations = {
                key: finalized_internal_links[key]
                for key in finalized_internal_links
                if key not in original_internal_links and key[0] in allowed_relation_types
            }

            if new_relations:
                updated_synsets.append({
                    'LEMMA': lemma,
                    'SENSE': sense,
                    'ID': synset_id,
                    'NEW_RELATIONS': new_relations  # Store relations as a dictionary
                })
                for rel_key in new_relations.keys():
                    target_id = rel_key[3]  # This is the ID of the TARGET_WM
                    newly_added_relations_info.append((rel_key[0], rel_key[1], rel_key[2], lemma, sense, target_id, synset_id))

    return updated_synsets, new_synsets, new_word_meanings_info, newly_added_relations_info

updated_synsets, new_synsets, new_word_meanings_info, newly_added_relations_info = identify_updates_in_iwn(ItalWN, IWN_post_mm)

# Creating the structured output
structured_relations = []

for rel_info in newly_added_relations_info:
    rel_type, target_lemma, target_sense, source_lemma, source_sense, target_id, source_synset_id = rel_info

    # Check if the source synset already exists in the structured output
    existing_entry = next((entry for entry in structured_relations if entry['WORD_MEANING_ID'] == source_synset_id), None)

    if existing_entry:
        # Add the new relation to the existing entry
        existing_entry['RELATIONS'].append({
            'RELATION_TYPE': rel_type,
            'TARGET_WM_ID': target_id,
            'TARGET_WM_LEMMA': target_lemma,
            'SENSE': target_sense
        })
    else:
        # Create a new entry if it doesn't exist
        structured_relations.append({
            'WORD_MEANING_ID': source_synset_id,
            'LITERAL': source_lemma,
            'SENSE': source_sense,
            'RELATIONS': [{
                'RELATION_TYPE': rel_type,
                'TARGET_WM_ID': target_id,
                'TARGET_WM_LEMMA': target_lemma,
                'SENSE': target_sense
            }]
        })


# Display new entries
print("New Synsets:")
for lemma, sense in new_word_meanings_info:
    print(f"Lemma: {lemma}, Sense: {sense}")

# Display newly added relations
print("\nNewly Added Relations:")
for rel_type, target_lemma, target_sense, lemma, sense, target_id, synset_id in newly_added_relations_info:
    print(f"Relation Type: {rel_type}, Target Lemma: {target_lemma}, Target Sense: {target_sense}, Synset Lemma: {lemma}, Synset Sense: {sense}, Target ID: {target_id}, Source Synset ID: {synset_id}")


In [ ]:
def find_glosses_for_new_synsets(xml_file, new_synsets):
    # Parse the XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Store results
    results = []

    # Iterate over synsets in the new_synsets list
    for synset in new_synsets:
        original_id = synset["ID"]
        numeric_id = original_id.split("#")[1]  # Strip anything before #

        # Find the <WORD_MEANING> node with the matching ID
        word_meaning = root.find(f".//WORD_MEANING[@ID='{original_id}']")
        if word_meaning is None:
            continue

        # Check if the <GLOSS> tag is empty
        gloss = word_meaning.find("GLOSS")
        if gloss is not None and (gloss.text is None or gloss.text.strip() == ""):
            lemma = synset["LEMMA"]

            # Look for this ID in <TARGET_WM> entries
            retrieved_gloss = None
            for target in root.findall(f".//TARGET_WM[@ID='{numeric_id}']"):
                retrieved_gloss = target.get("GLOSS")
                if retrieved_gloss:  # If a GLOSS attribute is found, stop searching
                    break

            # Add the result
            results.append({
                "id": numeric_id,
                "lemma": lemma,
                "retrieved_gloss": retrieved_gloss if retrieved_gloss else "No gloss found"
            })
    
    return results

def display_results(results):
    # Print results in a readable format
    print(f"{'ID':<10} {'Lemma':<30} {'Retrieved Gloss':<80}")
    print("-" * 120)
    for result in results:
        print(f"{result['id']:<10} {result['lemma']:<30} {result['retrieved_gloss']:<80}")

def update_glosses_in_xml(xml_file, results, output_file):
    # Parse the XML file
    tree = ET.parse(xml_file)
    root = tree.getroot()

    # Debugging list to track updates
    debugging_updates = []

    # Iterate through results to update the gloss
    for result in results:
        numeric_id = result["id"]
        lemma = result["lemma"]
        retrieved_gloss = result["retrieved_gloss"]

        # Find the <WORD_MEANING> node with the matching ID
        word_meaning = root.find(f".//WORD_MEANING[@ID='N#{numeric_id}']")
        if word_meaning is None:
            continue  # Skip if no matching node is found

        # Find the <GLOSS> node
        gloss_node = word_meaning.find("GLOSS")
        if gloss_node is not None and retrieved_gloss != "No gloss found":
            # Update the <GLOSS> node text with the retrieved gloss
            gloss_node.text = retrieved_gloss

            # Add to the debugging list
            debugging_updates.append({
                "id": numeric_id,
                "lemma": lemma,
                "added_gloss": retrieved_gloss
            })

    # Save the updated XML
    tree.write(output_file, encoding="utf-8", xml_declaration=True)

    # Print debugging updates
    print("\nUpdated Glosses:")
    for update in debugging_updates:
        print(f"ID: {update['id']}, Lemma: {update['lemma']}, Added Gloss: {update['added_gloss']}")

# Example usage
xml_file = IWN_post_mm  # Replace with your XML file path
output_file = IWN_mm_w_glosses  # Replace with your desired output XML file

results = find_glosses_for_new_synsets(xml_file, new_synsets)
display_results(results)
update_glosses_in_xml(xml_file, results, output_file)


In [ ]:
iwn_synsets_with_plugins = {}
target_mariterm_id = None
synset_relations_storage = []

def load_csv_data(csv_file_path):
    reference = {}
    with open(csv_file_path, mode='r', newline='', encoding='utf-8') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=';')
        for row in reader:
            iwn_lemma = row['Literal Lemma']
            iwn_sense = row.get('IWN sense', 'N/A')
            mari_lemma = row['Literal Lemma']
            mari_sense = row.get('MariT sense', 'N/A')
            reference[(iwn_lemma, iwn_sense)] = (mari_lemma, mari_sense)
    return reference

def find_mariterm_synset(root, lemma, sense):
    """Utility function to find a MariTerm synset by lemma and sense."""
    for wm in root.findall('.//WORD_MEANING'):
        literal = wm.find('.//LITERAL')
        if literal is not None and literal.get('LEMMA') == lemma and literal.get('SENSE') == sense:
            return wm
    return None

def insert_after(parent, new_element, after_element):
    """Inserts new_element in parent right after after_element."""
    found = False
    for i, elem in enumerate(parent):
        if elem == after_element:
            found = True
            parent.insert(i + 1, new_element)
            break
    if not found:
        parent.append(new_element)

def ensure_plugin_links(wm):
    """Ensure the <PLUG-IN_LINKS/> tag is present for synsets that don't have it."""
    plugin_links = wm.find('.//PLUG-IN_LINKS')
    if plugin_links is None:
        internal_links = wm.find('.//INTERNAL_LINKS')
        empty_plugin_links = ET.Element('PLUG-IN_LINKS')
        if internal_links is not None:
            insert_after(wm, empty_plugin_links, internal_links)
        else:
            wm.append(empty_plugin_links)
        print(f"Added empty PLUG-IN_LINKS for Lemma={wm.find('.//LITERAL').get('LEMMA')}, Sense={wm.find('.//LITERAL').get('SENSE')}")

def add_plugins(iwn_root, mariterm_filtered_root, mariterm_original_root, reference, newly_added_relations_info):
    relation_type_mapping = {
        "plug-synonym": "1",
        "plug-near_synonym": "2",
        "plug-xpos_near_synonym": "2",
        "plug-has_hyperonym": "3",
        "plug-has_xpos_hyperonym": "3",
        "plug-has_hyponym": "4",
        "plug-has_xpos_hyponym": "4",
    }

    for wm in iwn_root.findall('.//WORD_MEANING'):
        lemma_element = wm.find('.//LITERAL')
        lemma = lemma_element.get('LEMMA') if lemma_element is not None else None
        sense = lemma_element.get('SENSE') if lemma_element is not None else None

        if not lemma or not sense:
            continue

        mari_synset = reference.get((lemma, sense))
        if not mari_synset:
            ensure_plugin_links(wm)
            continue

        mari_lemma, mari_sense = mari_synset
        mariterm_synset = find_mariterm_synset(mariterm_filtered_root, mari_lemma, mari_sense)
        if mariterm_synset is None:
            mariterm_synset = find_mariterm_synset(mariterm_original_root, mari_lemma, mari_sense)
        if mariterm_synset is not None:
            mariterm_id = mariterm_synset.get('ID')
            literals = [lit.get('LEMMA') for lit in mariterm_synset.findall('.//LITERAL')]
            literals_str = ','.join(literals)

            plugin_links = wm.find('.//PLUG-IN_LINKS')
            if plugin_links is None:
                plugin_links = ET.Element('PLUG-IN_LINKS')
                internal_links = wm.find('.//INTERNAL_LINKS')
                if internal_links is not None:
                    insert_after(wm, plugin_links, internal_links)
                else:
                    wm.append(plugin_links)
            # Track the synsets with plugin links
            iwn_id = wm.get('ID')  # Get ID from WORD_MEANING
            first_literal_lemma = lemma  # Assuming lemma is defined earlier
            iwn_synsets_with_plugins[iwn_id] = first_literal_lemma


            if not any(rel['RELATION_TYPE'] == 'plug-synonym' and rel['TARGET_WM_ID'] == f"{mariterm_id}#{literals_str}" 
                    for entry in synset_relations_storage if entry['WORD_MEANING_ID'] == mariterm_id 
                    for rel in entry['RELATIONS']):
                # Proceed with adding plug-synonym only if it doesn't already exist
                plug_synonym = ET.SubElement(plugin_links, 'RELATION', TYPE="plug-synonym", ID="1", INV_ID="1")
                ET.SubElement(plug_synonym, 'TARGET_WM', ID=f"{mariterm_id}#{literals_str}")
                print(f"Added PLUG-IN LINK with target {mariterm_id}#{literals_str} for Lemma={lemma}, Sense={sense}.")
                
                # Create the plug-synonym entry to add to synset_relations_storage
                plug_synonym_entry = {
                    'RELATION_TYPE': 'plug-synonym',
                    'ID': '1',
                    'PART_OF_SPEECH': mariterm_synset.get('PART_OF_SPEECH'),
                    'TARGET_WM_ID': f"{mariterm_id}#{literals_str}",
                    'TARGET_WM_LEMMA': lemma,
                    'LITERALS': [lemma]  # Store the literal as a list
                }

                # Check if there's already an entry for the WORD_MEANING_ID in synset_relations_storage
                existing_entry = next((entry for entry in synset_relations_storage if entry['WORD_MEANING_ID'] == mariterm_id), None)

                if existing_entry:
                    # Check if the relation already exists
                    if not any(rel['RELATION_TYPE'] == 'plug-synonym' and rel['TARGET_WM_ID'] == plug_synonym_entry['TARGET_WM_ID'] for rel in existing_entry['RELATIONS']):
                        existing_entry['RELATIONS'].append(plug_synonym_entry)
                else:
                    synset_relations_storage.append({
                        'WORD_MEANING_ID': mariterm_id,
                        'FIRST LITERAL': lemma,
                        'RELATIONS': [plug_synonym_entry]  # New entry for new relations
                    })

            # Process only the relevant relations for this synset
            for relation_info in newly_added_relations_info:
                rel_type, target_lemma, target_sense, src_lemma, src_sense, target_id, synset_id = relation_info

                if src_lemma == lemma and src_sense == sense:
                    print(f"Processing new relation: Type={rel_type}, Target Lemma={target_lemma}, Target Sense={target_sense}")

                    if rel_type in allowed_relation_types:
                        print(f"Relation type {rel_type} is eligible for plug-in.")

                        mari_rel_synset = find_mariterm_synset(mariterm_filtered_root, target_lemma, target_sense)
                        if mari_rel_synset is None:
                            print(f"Target Lemma={target_lemma}, Sense={target_sense} not found in filtered MariTerm. Checking original MariTerm.")
                            mari_rel_synset = find_mariterm_synset(mariterm_original_root, target_lemma, target_sense)

                        if mari_rel_synset is not None:
                            part_of_speech = mari_rel_synset.get('PART_OF_SPEECH')
                            id_part = mari_rel_synset.get('ID')

                            if id_part.startswith(f"{part_of_speech}#"):
                                target_mariterm_id = id_part
                            else:
                                target_mariterm_id = f"{part_of_speech}#{id_part}"

                            word_meaning_synset = find_mariterm_synset(mariterm_original_root, target_mariterm_id.split('#')[1], target_sense)

                            if word_meaning_synset is not None:
                                literals = [lit.get('LEMMA') for lit in word_meaning_synset.findall('.//LITERAL')]
                                literals_str = ','.join(literals)
                                print(f"Retrieved literals: {literals_str}")
                            else:
                                literals_str = target_lemma
                                print(f"Fallback to using target lemma: {literals_str}")

                            plug_relation_type = f"plug-{rel_type}"
                            plug_relation_id = relation_type_mapping[plug_relation_type]
                            plug_relation = ET.SubElement(plugin_links, 'RELATION', TYPE=plug_relation_type, ID=plug_relation_id, INV_ID=plug_relation_id)

                            # Create a new entry for each relation type
                            relation_entry = {
                                'RELATION_TYPE': plug_relation_type,  # Updated to use the correct plug relation type
                                'ID': plug_relation_id,
                                'PART_OF_SPEECH': mariterm_synset.get('PART_OF_SPEECH'),
                                'TARGET_WM_ID': f"{mariterm_id}#{literals_str}",
                                'TARGET_WM_LEMMA': literals[0],
                                'LITERALS': literals
                            }

                            # Check if there's already an entry for the WORD_MEANING_ID in synset_relations_storage
                            existing_entry = next((entry for entry in synset_relations_storage if entry['WORD_MEANING_ID'] == mariterm_id), None)
                            if existing_entry:
                                existing_entry['RELATIONS'].append(relation_entry)
                            else:
                                synset_relations_storage.append({
                                    'WORD_MEANING_ID': mariterm_id,
                                    'FIRST LITERAL': first_literal_lemma,
                                    'RELATIONS': [relation_entry]  # New entry for new relations
                                })

                            if ' ' in literals_str:
                                literals_str = literals_str.replace(' ', '_')

                            ET.SubElement(plug_relation, 'TARGET_WM', ID=f"{target_mariterm_id}#{literals_str}")

                        else:
                            print(f"Target Lemma={target_lemma}, Sense={target_sense} not found in MariTerm.")
                    else:
                        print(f"Relation type {rel_type} is not eligible for plug-in.")
    return synset_relations_storage
                        
def remove_duplicate_plugins(iwn_root):
    for wm in iwn_root.findall('.//WORD_MEANING'):
        plugin_links = wm.find('.//PLUG-IN_LINKS')
        if plugin_links is not None:
            # Use a set to track unique relations
            unique_relations = set()
            relations_to_remove = []

            for relation in plugin_links.findall('.//RELATION'):
                rel_type = relation.get('TYPE')
                rel_id = relation.get('ID')
                target_wm = relation.find('.//TARGET_WM').get('ID')

                # Normalize the TARGET_WM ID by replacing spaces with underscores
                normalized_target_wm = target_wm.replace(' ', '_')

                # Create a tuple that uniquely identifies a relation (normalize target_wm)
                relation_key = (rel_type, rel_id, normalized_target_wm)

                if relation_key in unique_relations:
                    # Mark the relation for removal if it's a duplicate
                    relations_to_remove.append(relation)
                else:
                    # Add the normalized relation key to the set
                    unique_relations.add(relation_key)

            # Remove all duplicate relations
            for relation in relations_to_remove:
                plugin_links.remove(relation)

            if relations_to_remove:
                print(f"Removed {len(relations_to_remove)} duplicate relations for Lemma={wm.find('.//LITERAL').get('LEMMA')}, Sense={wm.find('.//LITERAL').get('SENSE')}")

def sort_plugin_links(plugin_links):
    # Extract relations
    relations = plugin_links.findall('.//RELATION')

    # Sort relations by ID (numerically) and then by the TARGET_WM attribute (alphabetically)
    relations.sort(key=lambda r: (int(r.get('ID')), r.find('.//TARGET_WM').get('ID')))

    # Clear existing RELATION elements and re-add them in the sorted order
    for rel in plugin_links.findall('.//RELATION'):
        plugin_links.remove(rel)

    for rel in relations:
        plugin_links.append(rel)

def save_pretty_xml(tree, xml_file_path):
    # Pretty-print the XML with proper formatting
    rough_string = ET.tostring(tree.getroot(), 'utf-8')
    reparsed = minidom.parseString(rough_string)

    # Custom cleanup to remove excessive whitespace
    pretty_xml = "\n".join([line for line in reparsed.toprettyxml(indent="    ").splitlines() if line.strip()])

    with open(xml_file_path, 'w', encoding='utf-8') as f:
        f.write(pretty_xml)
    print(f"Pretty XML file saved to {xml_file_path}")

# Load the necessary files
iwn_root = ET.parse(IWN_mm_w_glosses).getroot()
mariterm_filtered_root = ET.parse(filt_mart).getroot()
mariterm_original_root = ET.parse(MariT).getroot()

# Load the CSV data
reference = load_csv_data(breakdown_csv)

# After adding relations to the plugin_links
add_plugins(iwn_root, mariterm_filtered_root, mariterm_original_root, reference, newly_added_relations_info)

# Sort the relations inside each plugin_links
for wm in iwn_root.findall('.//WORD_MEANING'):
    plugin_links = wm.find('.//PLUG-IN_LINKS')
    if plugin_links is not None:
        sort_plugin_links(plugin_links)

# Remove any duplicate relations
remove_duplicate_plugins(iwn_root)

# Ensure all synsets have a <PLUG-IN_LINKS/> section, even if it's empty
ensure_plugin_links(iwn_root)

# Save the modified XML with proper formatting
tree = ET.ElementTree(iwn_root)
tree.write(finalized_IWN, encoding='utf-8', xml_declaration=True)
save_pretty_xml(tree, finalized_IWN)

In [ ]:
def update_plugin_links(iwn_root, mariterm_root, synset_relations_storage, iwn_synsets_with_plugins, structured_relations):
    valid_wm_ids = {synset['WORD_MEANING_ID'] for synset in synset_relations_storage}

    for word_meaning in mariterm_root.findall(".//WORD_MEANING"):
        wm_id = word_meaning.get('ID')

        # Remove existing PLUG-IN_LINKS if present
        existing_plugin_links = word_meaning.findall("PLUG-IN_LINKS")
        for plugin in existing_plugin_links:
            word_meaning.remove(plugin)
            print(f"Removed existing PLUG-IN_LINKS node for WM ID {wm_id}")

        # Find corresponding synset from synset_relations_storage
        corresponding_synset = next((synset for synset in synset_relations_storage if synset['WORD_MEANING_ID'] == wm_id), None)

        if corresponding_synset:
            plugin_links_node = ET.Element("PLUG-IN_LINKS")
            added_relations = set()  # Track added relations to avoid duplicates
            first_literal_lemma = word_meaning.find('.//LITERAL').get('LEMMA')
            current_pos = word_meaning.get('PART_OF_SPEECH')

            # Iterate through all relations in the corresponding synset
            for relation in corresponding_synset['RELATIONS']:
                # Check if the current first literal matches the relation's target literal
                for iwn_id, literal in iwn_synsets_with_plugins.items():
                    if first_literal_lemma.lower() in literal.lower():
                        target_wm_id = f"{iwn_id}#{first_literal_lemma}"

                        # Fetch all literals associated with the ID
                        iwn_entry = iwn_root.find(f".//WORD_MEANING[@ID='{iwn_id}']")
                        if iwn_entry is not None:
                            additional_literals = [lit.get('LEMMA') for lit in iwn_entry.findall('.//LITERAL')]
                            all_literals = list(set([first_literal_lemma]))  # Avoid duplicates in literals

                            for lit in additional_literals:
                                if lit not in all_literals:
                                    all_literals.append(lit)

                            target_wm_id = f"{iwn_id}#" + ','.join(all_literals)

                        target_wm_pos = iwn_entry.get('PART_OF_SPEECH') if iwn_entry is not None else None

                        # Add the relation if it's valid and not already added
                        relation_type = relation.get('RELATION_TYPE', '')
                        if relation_type and target_wm_id not in added_relations and target_wm_id != "" and (target_wm_pos == current_pos):
                            # Ensure relation type is prefixed with 'plug-'
                            if not relation_type.startswith('plug-'):
                                relation_type = f"plug-{relation_type}"

                            # Add the relation to the PLUG-IN_LINKS node
                            relation_node = ET.SubElement(plugin_links_node, "RELATION", {
                                "TYPE": relation_type,
                                "ID": relation['ID'],
                                "INV_ID": relation['ID']
                            })
                            ET.SubElement(relation_node, "TARGET_WM", {"ID": target_wm_id})
                            added_relations.add(target_wm_id)
                            print(f"Added relation {relation['RELATION_TYPE']} for {first_literal_lemma} with ID {target_wm_id}")

            # Insert the PLUG-IN_LINKS node if relations were added
            if added_relations:
                insert_plugin_node(word_meaning, plugin_links_node)
            else:
                # If no relations were added, insert an empty PLUG-IN_LINKS node
                print(f"No relations found for WM ID {wm_id}. Creating an empty PLUG-IN_LINKS node.")
                empty_plugin_links_node = ET.Element("PLUG-IN_LINKS")
                insert_plugin_node(word_meaning, empty_plugin_links_node)

        else:
            # No corresponding synset found, create an empty PLUG-IN_LINKS node
            print(f"No corresponding synset found for WM ID {wm_id}. Creating an empty PLUG-IN_LINKS node.")
            empty_plugin_links_node = ET.Element("PLUG-IN_LINKS")
            insert_plugin_node(word_meaning, empty_plugin_links_node)


def insert_plugin_node(word_meaning, plugin_links_node):
    inserted = False
    internal_links_node = word_meaning.find("INTERNAL_LINKS")
    eq_links_node = word_meaning.find("EQ_LINKS")

    # Insert after INTERNAL_LINKS if it exists
    if internal_links_node is not None:
        for i, child in enumerate(list(word_meaning)):
            if child is internal_links_node:
                word_meaning.insert(i + 1, plugin_links_node)
                inserted = True
                break

    # Otherwise, insert before EQ_LINKS if INTERNAL_LINKS is not found
    if not inserted and eq_links_node is not None:
        for i, child in enumerate(list(word_meaning)):
            if child is eq_links_node:
                word_meaning.insert(i, plugin_links_node)
                inserted = True
                break

    # If neither are found, just append the PLUG-IN_LINKS node at the end
    if not inserted:
        word_meaning.append(plugin_links_node)


# Call the function with the appropriate arguments
update_plugin_links(iwn_root, mariterm_original_root, synset_relations_storage, iwn_synsets_with_plugins, structured_relations)

# Save the updated MariTerm XML to a new file
m_tree = ET.ElementTree(mariterm_original_root)
save_pretty_xml(m_tree, finalized_MariT)


def insert_plugin_node(word_meaning, plugin_links_node):
    """
    Helper function to insert the PLUG-IN_LINKS node in the correct position.
    """
    inserted = False
    internal_links_node = word_meaning.find("INTERNAL_LINKS")
    eq_links_node = word_meaning.find("EQ_LINKS")

    # Insert after INTERNAL_LINKS if it exists
    if internal_links_node is not None:
        for i, child in enumerate(list(word_meaning)):
            if child is internal_links_node:
                word_meaning.insert(i + 1, plugin_links_node)
                inserted = True
                break

    # Otherwise, insert before EQ_LINKS if INTERNAL_LINKS is not found
    if not inserted and eq_links_node is not None:
        for i, child in enumerate(list(word_meaning)):
            if child is eq_links_node:
                word_meaning.insert(i, plugin_links_node)
                inserted = True
                break

    # If neither are found, just append the PLUG-IN_LINKS node at the end
    if not inserted:
        word_meaning.append(plugin_links_node)

# Call the function with the appropriate arguments
update_plugin_links(iwn_root, mariterm_original_root, synset_relations_storage, iwn_synsets_with_plugins, structured_relations)

# Save the updated MariTerm XML to a new file
m_tree = ET.ElementTree(mariterm_original_root)
save_pretty_xml(m_tree, finalized_MariT)  